# Core — Authentication & Response Parsing

> Shared utilities for Snowflake MCP Server and Cortex Agent interactions

In [ ]:
#| default_exp core

In [ ]:
#| export
import base64
import hashlib
import json
import os
import re
import time
import warnings
from dataclasses import dataclass, field
from typing import Any, Optional, Generator, Callable

import jwt
import requests
import pandas as pd
from cryptography.hazmat.primitives import serialization
from dotenv import load_dotenv

load_dotenv()

## Configuration

Credentials are loaded lazily via `get_config()`. A `RuntimeError` is raised listing all missing variables at once, rather than failing one-by-one at import time.

In [ ]:
#| export
def _env_or_default(name: str, default: str) -> str:
    """Return env var value if set and non-empty, otherwise default."""
    value = os.environ.get(name)
    return value if value else default


def get_config() -> dict:
    """Load and validate Snowflake configuration from environment variables.

    Returns a dict with keys: account, user, role, private_key_path,
    database, agent_schema, mcp_schema, mcp_server_name.
    Required vars raise RuntimeError if missing; optional vars have defaults.
    """
    required = [
        "SNOWFLAKE_ACCOUNT",
        "SNOWFLAKE_USER",
        "SNOWFLAKE_ROLE",
        "SNOWFLAKE_PRIVATE_KEY_PATH",
    ]
    missing = [k for k in required if not os.environ.get(k)]
    if missing:
        raise RuntimeError(
            f"Missing required environment variables: {', '.join(missing)}. "
            f"Copy .env.example to .env and fill in your values."
        )
    return {
        "account": os.environ["SNOWFLAKE_ACCOUNT"],
        "user": os.environ["SNOWFLAKE_USER"],
        "role": os.environ["SNOWFLAKE_ROLE"],
        "private_key_path": os.path.expanduser(os.environ["SNOWFLAKE_PRIVATE_KEY_PATH"]),
        "database": _env_or_default("SNOWFLAKE_DATABASE", "AM_SKI_RESORT"),
        "agent_schema": _env_or_default("SNOWFLAKE_AGENT_SCHEMA", "AGENTS"),
        "mcp_schema": _env_or_default("SNOWFLAKE_MCP_SCHEMA", "MCP_SERVERS"),
        "mcp_server_name": _env_or_default("SNOWFLAKE_MCP_SERVER", "ski_resort_mcp"),
    }

## JWT Authentication

Snowflake's REST APIs (MCP, Cortex Agent) authenticate via RSA key-pair JWTs. The `JWTGenerator` class handles private key loading, public key fingerprint calculation, and token caching with automatic refresh.

In [ ]:
#| export
class JWTGenerator:
    """Generate Snowflake-compatible JWTs using RSA key-pair authentication.

    Tokens are cached and automatically refreshed when within 60 seconds of expiry.
    The public key fingerprint is computed once from the private key on first use.
    """
    def __init__(self, account: str, user: str, private_key_path: str):
        self.account = account.upper().replace("-", "_")
        self.user = user.upper()
        self.private_key_path = private_key_path
        self._private_key = None
        self._public_key_fp: Optional[str] = None
        self._token: Optional[str] = None
        self._token_exp: int = 0

    def _load_private_key(self):
        if self._private_key is None:
            with open(self.private_key_path, "rb") as f:
                self._private_key = serialization.load_pem_private_key(
                    f.read(), password=None
                )
            pub_bytes = self._private_key.public_key().public_bytes(
                serialization.Encoding.DER,
                serialization.PublicFormat.SubjectPublicKeyInfo,
            )
            self._public_key_fp = "SHA256:" + base64.b64encode(
                hashlib.sha256(pub_bytes).digest()
            ).decode()
        return self._private_key

    def get_token(self) -> str:
        """Return a valid JWT, refreshing if needed."""
        now = int(time.time())
        if self._token and now < self._token_exp - 60:
            return self._token
        private_key = self._load_private_key()
        qualified = f"{self.account}.{self.user}"
        exp = now + 3600
        payload = {
            "iss": f"{qualified}.{self._public_key_fp}",
            "sub": qualified,
            "iat": now,
            "exp": exp,
        }
        self._token = jwt.encode(payload, private_key, algorithm="RS256")
        self._token_exp = exp
        return self._token

In [ ]:
#| export
class SnowflakeSession:
    """Holds Snowflake connection config, JWT generator, and header factory.

    The primary entry point for all authenticated Snowflake REST API calls.
    Create one per account/user combination; reuse it across requests.
    """

    def __init__(
        self,
        account: str,
        user: str,
        role: str,
        private_key_path: str,
        database: str = "AM_SKI_RESORT",
        agent_schema: str = "AGENTS",
        mcp_schema: str = "MCP_SERVERS",
        mcp_server_name: str = "ski_resort_mcp",
    ):
        self.account = account
        self.user = user
        self.role = role
        self.private_key_path = private_key_path
        self.database = database
        self.agent_schema = agent_schema
        self.mcp_schema = mcp_schema
        self.mcp_server_name = mcp_server_name
        self.host = f"https://{account}.snowflakecomputing.com"
        self.jwt_gen = JWTGenerator(account, user, private_key_path)

    @classmethod
    def from_env(cls) -> "SnowflakeSession":
        """Create a session from environment variables (via get_config())."""
        return cls(**get_config())

    def get_headers(self, accept: str = "application/json") -> dict:
        """Return HTTP headers for Snowflake REST API calls with a fresh JWT."""
        return {
            "Authorization": f"Bearer {self.jwt_gen.get_token()}",
            "X-Snowflake-Authorization-Token-Type": "KEYPAIR_JWT",
            "X-Snowflake-Role": self.role,
            "Content-Type": "application/json",
            "Accept": accept,
        }


_session: SnowflakeSession | None = None


def default_session() -> SnowflakeSession:
    """Return the default session, creating it lazily from env vars on first call.

    The session is cached for the lifetime of the kernel/process.
    Changing environment variables after the first call will not reconfigure
    the existing session. Call ``reset_default_session()`` to force re-creation.
    """
    global _session
    if _session is None:
        _session = SnowflakeSession.from_env()
    return _session


def reset_default_session() -> None:
    """Clear the cached default session, forcing re-creation on next access."""
    global _session
    _session = None

In [ ]:
import os
import mcp_ski_resort.core as _core_mod

_saved = {k: os.environ.pop(k, None) for k in [
    "SNOWFLAKE_DATABASE", "SNOWFLAKE_AGENT_SCHEMA",
    "SNOWFLAKE_MCP_SCHEMA", "SNOWFLAKE_MCP_SERVER",
]}
_core_mod._session = None
try:
    s = SnowflakeSession.from_env()
    assert s.database == "AM_SKI_RESORT", f"expected AM_SKI_RESORT, got {s.database}"
    assert s.agent_schema == "AGENTS"
    assert s.mcp_schema == "MCP_SERVERS"
    assert s.mcp_server_name == "ski_resort_mcp"
    print("defaults OK")

    os.environ["SNOWFLAKE_DATABASE"] = "CUSTOM_DB"
    os.environ["SNOWFLAKE_MCP_SERVER"] = "custom_mcp"
    s2 = SnowflakeSession.from_env()
    assert s2.database == "CUSTOM_DB"
    assert s2.mcp_server_name == "custom_mcp"
    assert s2.agent_schema == "AGENTS"
    print("env overrides OK")

    os.environ["SNOWFLAKE_DATABASE"] = ""
    os.environ["SNOWFLAKE_AGENT_SCHEMA"] = ""
    s3 = SnowflakeSession.from_env()
    assert s3.database == "AM_SKI_RESORT", f"empty string should fall back, got {s3.database}"
    assert s3.agent_schema == "AGENTS"
    print("empty-string fallback OK")
finally:
    for k, v in _saved.items():
        if v is not None:
            os.environ[k] = v
        else:
            os.environ.pop(k, None)
    _core_mod._session = None

In [ ]:
#| export
def get_headers(accept: str = "application/json") -> dict:
    """Return HTTP headers using the default session."""
    return default_session().get_headers(accept)

Let's verify the JWT generation works:


In [ ]:
#| eval: false
session = default_session()
token = session.jwt_gen.get_token()
print(f"Account:  {session.account}")
print(f"Host:     {session.host}")
print(f"User:     {session.user}")
print(f"JWT:      {token[:40]}...OK")

## Response Parsing

Snowflake returns result sets in a specific format where column names live in `resultSetMetaData.rowType[].name` (not the often-empty `columns` array). These helpers handle all MCP response formats: Analyst, Agent, Search, SQL, and GENERIC (UDF).

In [ ]:
#| export
def result_set_to_dataframe(rs: dict) -> pd.DataFrame:
    """Convert a Snowflake result_set dict to a pandas DataFrame.

    Checks `resultSetMetaData.rowType` first for column names (the reliable source),
    then falls back to the `columns` array, then dict keys, then positional columns.
    """
    data = rs.get("data", [])
    if not data:
        return pd.DataFrame()
    row_type = rs.get("resultSetMetaData", {}).get("rowType", [])
    if row_type:
        col_names = [r.get("name", f"col_{i}") for i, r in enumerate(row_type)]
        if not isinstance(data[0], dict) and len(col_names) != len(data[0]):
            warnings.warn(
                f"rowType has {len(col_names)} columns but data rows have {len(data[0])} elements; "
                f"falling back to positional columns"
            )
            return pd.DataFrame(data)
        return pd.DataFrame(data, columns=col_names)
    columns = rs.get("columns", [])
    col_names = [c if isinstance(c, str) else c.get("name", f"col_{i}") for i, c in enumerate(columns)]
    if col_names:
        if not isinstance(data[0], dict) and len(col_names) != len(data[0]):
            warnings.warn(
                f"columns has {len(col_names)} entries but data rows have {len(data[0])} elements; "
                f"falling back to positional columns"
            )
            return pd.DataFrame(data)
        return pd.DataFrame(data, columns=col_names)
    if isinstance(data[0], dict):
        return pd.DataFrame(data)
    return pd.DataFrame(data)

Here's a quick test with the `rowType` path (the common case for Cortex Agent responses):

In [ ]:
sample_rs = {
    "data": [["2024-2025", "15234", "8721"]],
    "resultSetMetaData": {
        "rowType": [
            {"name": "SKI_SEASON", "type": "TEXT"},
            {"name": "TOTAL_VISITS", "type": "NUMBER"},
            {"name": "UNIQUE_VISITORS", "type": "NUMBER"},
        ]
    },
    "columns": []
}
df = result_set_to_dataframe(sample_rs)
assert list(df.columns) == ["SKI_SEASON", "TOTAL_VISITS", "UNIQUE_VISITORS"]
assert len(df) == 1
df

In [ ]:
#| export
def extract_content(response: dict) -> list[dict]:
    """Extract the content array from an MCP JSON-RPC response."""
    return response.get("result", {}).get("content", [])


def extract_text(response: dict) -> str:
    """Extract concatenated text from an MCP JSON-RPC response.

    Only items with `type: text` are included. Non-text content types
    (e.g. image, resource) are silently skipped.
    """
    items = extract_content(response)
    return "\n".join(item.get("text", "") for item in items if item.get("type") == "text")


_FENCE_RE = re.compile(r'^```(?:json)?\s*\n?(.*?)\n?```$', re.DOTALL)


def try_parse_json(text: str):
    """Attempt to parse a string as JSON, stripping markdown fences if present.

    Returns the parsed object, or None on failure.
    """
    if not text:
        return None
    text = text.strip()
    m = _FENCE_RE.match(text)
    if m:
        text = m.group(1).strip()
    try:
        return json.loads(text)
    except (json.JSONDecodeError, TypeError):
        return None

In [ ]:
#| export
def parse_analyst_response(response: dict) -> dict:
    """Parse a Cortex Analyst MCP response into interpretations, SQL statements, and result_sets.

    Returns lists to handle multi-statement responses correctly.
    """
    text = extract_text(response)
    parsed = try_parse_json(text)
    if isinstance(parsed, list):
        interpretations = []
        sql_statements = []
        result_sets = []
        for item in parsed:
            if isinstance(item, dict):
                if "text" in item:
                    interpretations.append(item["text"])
                if "statement" in item:
                    sql_statements.append(item["statement"])
                if "resultSet" in item:
                    result_sets.append(item["resultSet"])
        return {
            "interpretation": interpretations[-1] if interpretations else None,
            "interpretations": interpretations,
            "sql": sql_statements[-1] if sql_statements else None,
            "sql_statements": sql_statements,
            "result_set": result_sets[-1] if result_sets else None,
            "result_sets": result_sets,
        }
    return {"raw": text}


def parse_agent_response(response: dict) -> dict:
    """Parse a Cortex Agent MCP response into answer, SQL queries, and result_sets."""
    text = extract_text(response)
    parsed = try_parse_json(text)
    if isinstance(parsed, dict):
        content_list = parsed.get("content", [])
        answer_parts = []
        sql_queries = []
        result_sets = []
        for item in content_list:
            if isinstance(item, dict):
                if "text" in item:
                    answer_parts.append(item.get("text", ""))
                tool_results = item.get("tool_results", {})
                if isinstance(tool_results, dict):
                    for tr in tool_results.get("content", []):
                        j = tr.get("json", {}) if isinstance(tr, dict) else {}
                        if j.get("sql"):
                            sql_queries.append(j["sql"])
                        if j.get("result_set"):
                            result_sets.append(j["result_set"])
        return {
            "answer": "\n".join(part for part in answer_parts if part),
            "sql_queries": sql_queries,
            "result_sets": result_sets,
            "raw": parsed,
        }
    return {"answer": text, "raw": text}


def parse_search_response(response: dict) -> pd.DataFrame:
    """Parse a Cortex Search MCP response into a DataFrame."""
    text = extract_text(response)
    parsed = try_parse_json(text)
    if isinstance(parsed, list):
        return pd.DataFrame(parsed)
    return pd.DataFrame()

## Agent SSE Streaming

The Cortex Agent REST API uses Server-Sent Events (SSE) for streaming responses. These helpers parse the SSE stream and normalize the various event types into a clean interface.

In [ ]:
#| export
def build_agent_messages(
    question: str, history: list[dict] | None = None
) -> list[dict]:
    """Build the messages payload for the Cortex Agent :run endpoint."""
    messages = []
    for m in (history or []):
        messages.append({
            "role": m["role"],
            "content": [{"type": "text", "text": m["content"]}],
        })
    messages.append({
        "role": "user",
        "content": [{"type": "text", "text": question}],
    })
    return messages

In [ ]:
#| export
def build_agent_run_payload(
    question: str,
    history: list[dict] | None = None,
    thread_id: str | None = None,
    parent_message_id: str | None = None,
) -> dict:
    """Build the complete JSON payload for the Cortex Agent ``:run`` endpoint.

    Two modes:

    - **Local-history mode** (default): ``history`` is formatted into the
      messages array. Conversation continuity is managed by the caller.
    - **Thread mode** (``thread_id`` provided): the server-side thread owns
      conversation continuity. ``history`` is **not** sent upstream -- pass
      it to ``AgentChat`` for local transcript/inspection only.
      ``parent_message_id`` defaults to ``"0"`` for the first message in
      a thread.

    When both ``history`` and ``thread_id`` are provided, thread mode wins
    for the upstream payload. The caller is responsible for keeping local
    history separately if desired.
    """
    if thread_id:
        return {
            "messages": build_agent_messages(question),
            "thread_id": thread_id,
            "parent_message_id": parent_message_id or "0",
        }
    return {"messages": build_agent_messages(question, history)}


In [ ]:
#| export
def stream_agent_sse(
    agent_name: str,
    question: str,
    history: list[dict] | None = None,
    session: SnowflakeSession | None = None,
    thread_id: str | None = None,
    parent_message_id: str | None = None,
) -> Generator[dict, None, None]:
    """POST to the Cortex Agent :run endpoint and yield parsed SSE events.

    Each yielded dict has ``{"event": str, "data": dict}``.
    Malformed JSON in SSE data lines yields a ``parse_error`` event instead
    of being silently dropped.
    """
    s = session or default_session()
    endpoint = (
        f"{s.host}/api/v2/databases/{s.database}"
        f"/schemas/{s.agent_schema}/agents/{agent_name}:run"
    )

    payload = build_agent_run_payload(question, history, thread_id, parent_message_id)

    with requests.post(endpoint, headers=s.get_headers(accept="text/event-stream"),
                       json=payload, stream=True, timeout=120) as resp:
        resp.raise_for_status()

        current_event: str | None = None
        data_buffer: list[str] = []

        for raw_line in resp.iter_lines(decode_unicode=True):
            if raw_line is None:
                continue
            line = raw_line

            if not line:
                if data_buffer and current_event is not None:
                    joined = "\n".join(data_buffer)
                    if joined == "[DONE]":
                        yield {"event": "done", "data": {}}
                        return
                    try:
                        data = json.loads(joined)
                        yield {"event": current_event, "data": data}
                    except json.JSONDecodeError as exc:
                        yield {
                            "event": "parse_error",
                            "data": {"raw_event": current_event, "raw_data": joined[:500], "error": str(exc)},
                        }
                current_event = None
                data_buffer = []
                continue

            if line.startswith("event:"):
                current_event = line[6:].strip()
                continue

            if line.startswith("data:"):
                data_buffer.append(line[5:].strip())
                continue

        if data_buffer and current_event is not None:
            joined = "\n".join(data_buffer)
            if joined == "[DONE]":
                yield {"event": "done", "data": {}}
                return
            try:
                data = json.loads(joined)
                yield {"event": current_event, "data": data}
            except json.JSONDecodeError as exc:
                yield {
                    "event": "parse_error",
                    "data": {"raw_event": current_event, "raw_data": joined[:500], "error": str(exc)},
                }


In [ ]:
#| export
def normalize_event(raw_event: str, data: dict, seen_tool_result: bool = False) -> list[dict]:
    """Translate raw Cortex Agent SSE events into clean normalized events.

    Returns a list of normalized ``{event, data}`` dicts.
    Event types: text, thinking, thinking_complete, status, tool, sql, table,
    chart, annotation, error, metadata, done, parse_error.

    Note on ``seen_tool_result``: Cortex Agents emit ``response.text.delta`` for both
    thinking text (before tool execution) and final answer text (after tool results
    come back). The ``seen_tool_result`` flag disambiguates: before any tool_result
    event, text deltas are classified as ``thinking``; after, they become ``text``.
    This is based on observed agent protocol behavior and may need updating if
    the SSE event contract changes.
    """
    results = []

    if raw_event == "response.text.delta":
        text = data.get("text")
        if text:
            evt = "text" if seen_tool_result else "thinking"
            results.append({"event": evt, "data": {"text": text}})

    elif raw_event == "response.thinking.delta":
        text = data.get("thinking") or data.get("text", "")
        if text:
            results.append({"event": "thinking", "data": {"text": text}})

    elif raw_event == "response":
        text = data.get("thinking") or data.get("text", "")
        if text:
            results.append({"event": "thinking_complete", "data": {"text": text}})

    elif raw_event == "response.status":
        msg = data.get("status_message") or data.get("message", "")
        if msg:
            results.append({"event": "status", "data": {"message": msg}})

    elif raw_event == "response.tool_use":
        name = data.get("name", "")
        clean = name.replace("cortex_analyst_text_to_sql__", "").replace("_", " ")
        if clean:
            results.append({"event": "tool", "data": {"name": clean}})

    elif raw_event == "response.tool_result":
        for item in data.get("content", []):
            j = item.get("json", {}) if isinstance(item, dict) else {}
            if j.get("sql"):
                results.append({"event": "sql", "data": {"sql": j["sql"]}})
            if j.get("result_set"):
                results.append({"event": "table", "data": j["result_set"]})

    elif raw_event == "response.chart":
        raw_spec = data.get("chart_spec")
        if raw_spec:
            try:
                spec = json.loads(raw_spec) if isinstance(raw_spec, str) else raw_spec
                results.append({"event": "chart", "data": spec})
            except json.JSONDecodeError:
                results.append({"event": "parse_error", "data": {"raw_event": raw_event, "raw_data": str(raw_spec)[:200]}})

    elif raw_event == "response.text.annotation":
        results.append({"event": "annotation", "data": data})

    elif raw_event == "response.error":
        msg = data.get("message") or data.get("error", "An error occurred")
        results.append({"event": "error", "data": {"message": msg}})

    elif raw_event == "metadata":
        inner = data.get("metadata", data)
        if "message_id" in inner:
            inner = {**inner, "message_id": str(inner["message_id"])}
        results.append({"event": "metadata", "data": inner})

    elif raw_event == "done":
        results.append({"event": "done", "data": {}})

    elif raw_event == "parse_error":
        results.append({"event": "parse_error", "data": data})

    return results


In [ ]:
#| export
@dataclass
class AgentResult:
    """Accumulated result from a Cortex Agent streaming call."""
    answer: str = ""
    thinking: list[str] = field(default_factory=list)
    sql_queries: list[str] = field(default_factory=list)
    result_sets: list[dict] = field(default_factory=list)
    dataframes: list[pd.DataFrame] = field(default_factory=list)
    chart_specs: list[dict] = field(default_factory=list)
    tools_used: list[str] = field(default_factory=list)
    statuses: list[str] = field(default_factory=list)
    errors: list[str] = field(default_factory=list)
    raw_events: list[dict] = field(default_factory=list)
    duration_seconds: float = 0.0
    thread_metadata: dict[str, Any] = field(default_factory=dict)


def _apply_normalized_event(
    result: AgentResult, etype: str, edata: dict, current_thinking: str
) -> str:
    """Apply a single normalized event to an AgentResult.

    Returns the updated ``current_thinking`` accumulator string.
    """
    if etype == "text":
        result.answer += edata.get("text", "")
    elif etype == "thinking":
        current_thinking += edata.get("text", "")
    elif etype == "thinking_complete":
        if current_thinking:
            result.thinking.append(current_thinking)
            current_thinking = ""
        text = edata.get("text", "")
        if text:
            result.thinking.append(text)
    elif etype == "status":
        result.statuses.append(edata.get("message", ""))
    elif etype == "tool":
        name = edata.get("name", "")
        if name not in result.tools_used:
            result.tools_used.append(name)
    elif etype == "sql":
        result.sql_queries.append(edata.get("sql", ""))
    elif etype == "table":
        result.result_sets.append(edata)
        result.dataframes.append(result_set_to_dataframe(edata))
    elif etype == "chart":
        result.chart_specs.append(edata)
    elif etype == "metadata":
        result.thread_metadata.update(edata)
    elif etype in ("error", "parse_error"):
        result.errors.append(edata.get("message", edata.get("error", "")))
    return current_thinking


def _finalize_agent_result(result: AgentResult, current_thinking: str) -> None:
    """Finalize accumulated agent state after all SSE events are processed.

    Flushes any remaining thinking text. For tool-less successful runs where
    all text was classified as thinking (the ``seen_tool_result`` heuristic),
    promotes thinking to answer so callers get a meaningful ``result.answer``.
    """
    if current_thinking:
        result.thinking.append(current_thinking)

    if (
        not result.answer
        and not result.tools_used
        and not result.errors
        and result.thinking
    ):
        result.answer = "\n".join(result.thinking)


def _default_reporter(etype: str, edata: dict):
    """Default verbose reporter for run_agent -- prints status to stdout."""
    if etype == "thinking":
        print(f"  [thinking] {edata.get('text', '')[:80]}...", end="\r")
    elif etype == "status":
        print(f"  [status] {edata.get('message', '')}")
    elif etype == "tool":
        print(f"  [tool] {edata.get('name', '')}")
    elif etype == "sql":
        print(f"  [sql] {edata.get('sql', '')[:100]}...")
    elif etype == "table":
        print(f"  [table] {edata.get('rows', '?')} rows x {edata.get('cols', '?')} cols")
    elif etype == "chart":
        print(f"  [chart] Vega-Lite spec received")
    elif etype == "error":
        print(f"  [ERROR] {edata.get('message', '')}")
    elif etype == "parse_error":
        print(f"  [PARSE ERROR] {edata.get('error', '')} on {edata.get('raw_event', '')}")


def _iter_raw_and_normalized_agent_events(
    agent_name: str,
    question: str,
    history: list[dict] | None = None,
    session: SnowflakeSession | None = None,
    thread_id: str | None = None,
    parent_message_id: str | None = None,
) -> Generator[tuple[dict, list[dict]], None, None]:
    """Yield ``(raw_event, normalized_events)`` tuples from a Cortex Agent stream.

    Private driver that owns the ``seen_tool_result`` state machine.
    All higher-level functions build on this to avoid duplicating
    normalization logic.
    """
    seen_tool_result = False
    for raw in stream_agent_sse(
        agent_name, question, history, session=session,
        thread_id=thread_id, parent_message_id=parent_message_id,
    ):
        raw_event = raw["event"]
        raw_data = raw.get("data", {})
        if raw_event == "response.tool_result":
            seen_tool_result = True
        normalized = normalize_event(raw_event, raw_data, seen_tool_result=seen_tool_result)
        yield raw, normalized


def iter_normalized_agent_events(
    agent_name: str,
    question: str,
    history: list[dict] | None = None,
    session: SnowflakeSession | None = None,
    thread_id: str | None = None,
    parent_message_id: str | None = None,
) -> Generator[dict, None, None]:
    """Stream and normalize Cortex Agent events in one step.

    Yields normalized ``{"event": str, "data": dict}`` dicts. Callers do
    not need to track ``seen_tool_result`` or call ``normalize_event()``
    themselves. This is the recommended entry point for building custom
    SSE proxies.
    """
    for _raw, normalized_items in _iter_raw_and_normalized_agent_events(
        agent_name, question, history, session=session,
        thread_id=thread_id, parent_message_id=parent_message_id,
    ):
        yield from normalized_items


def collect_agent_events(
    agent_name: str,
    question: str,
    history: list[dict] | None = None,
    session: SnowflakeSession | None = None,
    thread_id: str | None = None,
    parent_message_id: str | None = None,
) -> AgentResult:
    """Pure collector: stream agent events and accumulate into AgentResult.

    No printing or side effects. Use this when you want programmatic access only.
    """
    result = AgentResult()
    start = time.time()
    current_thinking = ""

    for raw, normalized_items in _iter_raw_and_normalized_agent_events(
        agent_name, question, history, session=session,
        thread_id=thread_id, parent_message_id=parent_message_id,
    ):
        result.raw_events.append(raw)
        for evt in normalized_items:
            current_thinking = _apply_normalized_event(
                result, evt["event"], evt["data"], current_thinking
            )

    _finalize_agent_result(result, current_thinking)
    result.duration_seconds = round(time.time() - start, 2)
    return result


def run_agent(
    agent_name: str,
    question: str,
    history: list[dict] | None = None,
    verbose: bool = True,
    reporter: Callable[[str, dict], None] | None = None,
    session: SnowflakeSession | None = None,
    thread_id: str | None = None,
    parent_message_id: str | None = None,
) -> AgentResult:
    """Call a Cortex Agent with streaming SSE and accumulate the full result.

    Args:
        agent_name: Name of the Cortex Agent to call.
        question: User question text.
        history: Optional conversation history.
        verbose: If True and no reporter given, prints progress to stdout.
        reporter: Optional callback ``(event_type, event_data) -> None`` for custom output.
            Callbacks fire after each event is applied to the AgentResult.
            For table events, edata is enriched: ``{"result_set": ..., "rows": N, "cols": N}``.
            For all other events, edata is passed through from normalize_event().
        session: Optional SnowflakeSession; uses default if None.
        thread_id: Optional Cortex thread identifier for server-side continuity.
        parent_message_id: Optional parent message for threading. Defaults to
            ``"0"`` when ``thread_id`` is provided.
    """
    cb = reporter or (_default_reporter if verbose else None)

    if verbose and cb is _default_reporter:
        print(f"Calling {agent_name} with: {question!r}")
        print("-" * 60)

    result = AgentResult()
    start = time.time()
    current_thinking = ""

    for raw, normalized_items in _iter_raw_and_normalized_agent_events(
        agent_name, question, history, session=session,
        thread_id=thread_id, parent_message_id=parent_message_id,
    ):
        result.raw_events.append(raw)
        for evt in normalized_items:
            etype = evt["event"]
            edata = evt["data"]
            current_thinking = _apply_normalized_event(
                result, etype, edata, current_thinking
            )
            if cb:
                if etype == "table":
                    df = result.dataframes[-1]
                    cb(etype, {"result_set": edata, "rows": len(df), "cols": len(df.columns)})
                else:
                    cb(etype, edata)

    _finalize_agent_result(result, current_thinking)
    result.duration_seconds = round(time.time() - start, 2)

    if verbose and cb is _default_reporter:
        print("-" * 60)
        print(f"Done in {result.duration_seconds}s | Tools: {result.tools_used} | SQL: {len(result.sql_queries)} | Tables: {len(result.dataframes)}")

    return result


def create_thread(
    session: SnowflakeSession | None = None,
    origin_application: str = "mcp_ski_resort",
) -> str:
    """Create a Cortex conversation thread. Returns the ``thread_id`` string.

    Only the thread identifier is returned. The full server response
    (which may include additional metadata) is not preserved.

    Raises ``KeyError`` if the response contains neither ``thread_id``
    nor ``id``.
    """
    s = session or default_session()
    resp = requests.post(
        f"{s.host}/api/v2/cortex/threads",
        headers=s.get_headers(),
        json={"origin_application": origin_application},
        timeout=30,
    )
    resp.raise_for_status()
    data = resp.json()
    tid = data.get("thread_id") or data.get("id")
    if tid is None:
        raise KeyError(f"Thread response missing thread_id and id: {list(data.keys())}")
    return str(tid)


class AgentChat:
    """Stateful conversation wrapper for Cortex Agents.

    Operates in two modes:

    - **Local-history mode** (default): conversation history is sent with
      each request. The caller owns continuity.
    - **Thread mode** (``thread_id`` provided): the server-side thread
      owns continuity. Local history is still recorded for inspection,
      debugging, and transcript export, but is **not** sent upstream.
      ``parent_message_id`` is tracked automatically from ``metadata``
      events.

    Lower-level control is available via ``run_agent()`` and
    ``stream_agent_sse()``.
    """

    def __init__(
        self,
        agent_name: str,
        session: SnowflakeSession | None = None,
        verbose: bool = True,
        reporter: Callable[[str, dict], None] | None = None,
        history: list[dict] | None = None,
        thread_id: str | None = None,
    ):
        self.agent_name = agent_name
        self.session = session
        self.verbose = verbose
        self.reporter = reporter
        self.history: list[dict] = list(history or [])
        self.results: list[AgentResult] = []
        self.thread_id = thread_id
        self._parent_message_id: str | None = None

    def ask(self, question: str, **kwargs) -> AgentResult:
        """Send a question; history/thread state is carried automatically."""
        result = run_agent(
            self.agent_name,
            question,
            history=self.history if not self.thread_id else None,
            verbose=kwargs.get("verbose", self.verbose),
            reporter=kwargs.get("reporter", self.reporter),
            session=self.session,
            thread_id=self.thread_id,
            parent_message_id=self._parent_message_id,
        )
        if result.thread_metadata.get("message_id"):
            self._parent_message_id = result.thread_metadata["message_id"]
        self.history.append({"role": "user", "content": question})
        assistant_text = result.answer or "\n".join(result.thinking)
        self.history.append({"role": "assistant", "content": assistant_text})
        self.results.append(result)
        return result

    @property
    def last(self) -> AgentResult | None:
        """Most recent result, or None if no questions asked yet."""
        return self.results[-1] if self.results else None

    def reset(self) -> "AgentChat":
        """Clear local transcript, results, and parent message cursor.

        Does **not** unset ``thread_id`` -- the chat stays in the same
        mode (local-history or thread). To switch modes, create a new
        ``AgentChat`` instance.
        """
        self.history.clear()
        self.results.clear()
        self._parent_message_id = None
        return self

    def __repr__(self) -> str:
        mode = f"thread={self.thread_id}" if self.thread_id else "local-history"
        return f"AgentChat({self.agent_name!r}, turns={len(self.results)}, {mode})"


In [ ]:
# ── Test _apply_normalized_event accumulation ──
r = AgentResult()
ct = ""

# 1. thinking -> thinking_complete flushes accumulated text
ct = _apply_normalized_event(r, "thinking", {"text": "Analyzing "}, ct)
ct = _apply_normalized_event(r, "thinking", {"text": "the data..."}, ct)
assert ct == "Analyzing the data..."
ct = _apply_normalized_event(r, "thinking_complete", {"text": ""}, ct)
assert r.thinking == ["Analyzing the data..."]
assert ct == ""

# 2. text event (post-tool-result) accumulates into answer
ct = _apply_normalized_event(r, "text", {"text": "The answer is 42."}, ct)
assert r.answer == "The answer is 42."

# 3. table event appends result_set and dataframe with correct shape
table_data = {
    "data": [["A", 1], ["B", 2], ["C", 3]],
    "resultSetMetaData": {
        "rowType": [{"name": "NAME"}, {"name": "VALUE"}]
    },
}
ct = _apply_normalized_event(r, "table", table_data, ct)
assert len(r.result_sets) == 1
assert len(r.dataframes) == 1
assert r.dataframes[0].shape == (3, 2)
assert list(r.dataframes[0].columns) == ["NAME", "VALUE"]

# 4. parse_error lands in errors list
ct = _apply_normalized_event(r, "parse_error", {"error": "bad json", "raw_event": "response.chart"}, ct)
assert r.errors == ["bad json"]

# 4b. metadata event stores in thread_metadata
ct = _apply_normalized_event(r, "metadata", {"message_id": "99"}, ct)
assert r.thread_metadata == {"message_id": "99"}

print("All _apply_normalized_event tests passed")

# ── Test event semantics: normalize_event + _apply + _finalize ──

# 5. Tool-less direct answer: all text.delta -> thinking, finalize promotes to answer
r2 = AgentResult()
ct2 = ""
for text in ["Hello, ", "how ", "can I help?"]:
    evts = normalize_event("response.text.delta", {"text": text}, seen_tool_result=False)
    for e in evts:
        ct2 = _apply_normalized_event(r2, e["event"], e["data"], ct2)
assert r2.answer == "", "before finalize, answer should be empty for tool-less flow"
assert ct2 == "Hello, how can I help?"
_finalize_agent_result(r2, ct2)
assert r2.answer == "Hello, how can I help?", f"finalize should promote thinking->answer, got {r2.answer!r}"
assert r2.thinking == ["Hello, how can I help?"]
print("tool-less direct answer OK")

# 6. Tool-using answer: text before tool -> thinking, text after -> answer
r3 = AgentResult()
ct3 = ""
seen = False
for e in normalize_event("response.text.delta", {"text": "Let me check..."}, seen_tool_result=False):
    ct3 = _apply_normalized_event(r3, e["event"], e["data"], ct3)
for e in normalize_event("response.tool_use", {"name": "query_db"}, seen_tool_result=False):
    ct3 = _apply_normalized_event(r3, e["event"], e["data"], ct3)
seen = True
tool_result_data = {"content": [{"json": {"sql": "SELECT 1", "result_set": {"data": [[1]], "columns": ["val"]}}}]}
for e in normalize_event("response.tool_result", tool_result_data, seen_tool_result=seen):
    ct3 = _apply_normalized_event(r3, e["event"], e["data"], ct3)
for e in normalize_event("response.text.delta", {"text": "The result is 1."}, seen_tool_result=True):
    ct3 = _apply_normalized_event(r3, e["event"], e["data"], ct3)
_finalize_agent_result(r3, ct3)
assert r3.answer == "The result is 1.", f"post-tool text should be answer, got {r3.answer!r}"
assert "Let me check..." in r3.thinking[0]
assert len(r3.sql_queries) == 1
assert len(r3.result_sets) == 1
assert r3.tools_used == ["query db"]
print("tool-using answer OK")

# 7. Error run: finalize should NOT promote thinking to answer
r4 = AgentResult()
ct4 = ""
for e in normalize_event("response.text.delta", {"text": "Starting..."}, seen_tool_result=False):
    ct4 = _apply_normalized_event(r4, e["event"], e["data"], ct4)
for e in normalize_event("response.error", {"message": "something broke"}, seen_tool_result=False):
    ct4 = _apply_normalized_event(r4, e["event"], e["data"], ct4)
_finalize_agent_result(r4, ct4)
assert r4.answer == "", "error run should not promote thinking to answer"
assert r4.errors == ["something broke"]
print("error run OK")

# 8. build_agent_messages constructs correct payload
msgs = build_agent_messages("hello", [{"role": "user", "content": "prior"}, {"role": "assistant", "content": "yes"}])
assert len(msgs) == 3
assert msgs[0]["role"] == "user"
assert msgs[0]["content"][0]["text"] == "prior"
assert msgs[2]["content"][0]["text"] == "hello"
print("build_agent_messages OK")

# ── build_agent_run_payload mode tests ──

# 9. Local-history mode
p1 = build_agent_run_payload("hello", [{"role": "user", "content": "prior"}])
assert "thread_id" not in p1
assert len(p1["messages"]) == 2

# 10. Thread mode — no history sent upstream
p2 = build_agent_run_payload("hello", thread_id="t-123")
assert p2["thread_id"] == "t-123"
assert p2["parent_message_id"] == "0"
assert len(p2["messages"]) == 1

# 11. Thread mode with explicit parent
p3 = build_agent_run_payload("hello", thread_id="t-123", parent_message_id="msg-5")
assert p3["parent_message_id"] == "msg-5"

# 12. history + thread_id — thread mode wins, history ignored upstream
p4 = build_agent_run_payload("hello", [{"role": "user", "content": "prior"}], thread_id="t-123")
assert len(p4["messages"]) == 1
assert p4["thread_id"] == "t-123"
print("build_agent_run_payload OK")

# ── Metadata event tests ──

# 13. normalize_event coerces message_id to string
evts = normalize_event("metadata", {"metadata": {"message_id": 12345}})
assert evts[0]["data"]["message_id"] == "12345"
assert evts[0]["event"] == "metadata"

# 14. normalize_event passes through string message_id
evts2 = normalize_event("metadata", {"metadata": {"message_id": "abc"}})
assert evts2[0]["data"]["message_id"] == "abc"

# 15. _apply stores metadata in thread_metadata
rm = AgentResult()
_apply_normalized_event(rm, "metadata", {"message_id": "42", "role": "assistant"}, "")
assert rm.thread_metadata == {"message_id": "42", "role": "assistant"}

# 16. Multiple metadata events merge
_apply_normalized_event(rm, "metadata", {"thread_id": "t-1"}, "")
assert rm.thread_metadata == {"message_id": "42", "role": "assistant", "thread_id": "t-1"}
print("metadata tests OK")

# ── AgentChat thread-mode tests (synthetic, no network) ──

def _fake_stream(agent_name, question, history=None, session=None,
                 thread_id=None, parent_message_id=None):
    """Fake stream_agent_sse that yields synthetic raw SSE events."""
    yield {"event": "response.text.delta", "data": {"text": f"Answer to: {question}"}}
    yield {"event": "metadata", "data": {"metadata": {"message_id": f"msg-{abs(hash(question)) % 1000}"}}}
    yield {"event": "done", "data": {}}

_real_stream = stream_agent_sse
try:
    globals()["stream_agent_sse"] = _fake_stream
    import mcp_ski_resort.core as _cm
    _cm.stream_agent_sse = _fake_stream

    chat = AgentChat("test_agent", verbose=False, thread_id="t-abc")

    # 17. repr shows thread mode
    assert "thread=t-abc" in repr(chat)

    # 18. First turn — parent starts as None
    r1 = chat.ask("question one")
    assert r1.answer == "Answer to: question one"
    assert r1.thread_metadata.get("message_id") is not None
    first_msg_id = r1.thread_metadata["message_id"]
    assert chat._parent_message_id == first_msg_id

    # 19. Local transcript grows
    assert len(chat.history) == 2
    assert chat.history[0] == {"role": "user", "content": "question one"}
    assert chat.history[1]["role"] == "assistant"

    # 20. Second turn — parent_message_id advances
    r2 = chat.ask("question two")
    second_msg_id = r2.thread_metadata["message_id"]
    assert chat._parent_message_id == second_msg_id
    assert first_msg_id != second_msg_id
    assert len(chat.history) == 4
    assert len(chat.results) == 2

    # 21. reset() clears transcript but keeps thread_id
    chat.reset()
    assert chat.thread_id == "t-abc"
    assert chat._parent_message_id is None
    assert len(chat.history) == 0
    assert len(chat.results) == 0

    # 22. last property
    assert chat.last is None
    r3 = chat.ask("after reset")
    assert chat.last is r3

    print("AgentChat thread-mode tests OK")
finally:
    globals()["stream_agent_sse"] = _real_stream
    _cm.stream_agent_sse = _real_stream

print("\nAll tests passed")


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()